## **Tasa Interes BD**

esta acion permite guardar en un SQL lite

In [3]:
import pandas as pd
from pathlib import Path
import sqlite3

In [4]:
data_tasas = pd.read_excel(rf"C:\Users\PC\Desktop\Proyectos\Proyectos_Py\9.Tasas_Interes\data\processed\tasa_interes_final.xlsx")

In [5]:
data_tasas["Bancos"].unique()

array(['BBVA', 'Bancom', 'Crédito', 'Pichincha', 'BIF', 'Scotiabank',
       'Citibank', 'Citibank.1', 'Interbank', 'Mibanco', 'GNB',
       'Falabella', 'Santander', 'Ripley', 'Alfin', 'ICBC',
       'Bank of China', 'BCI', 'Compartamos', 'Santander Cons. Bank',
       'Promedio', 'Bank of China.1'], dtype=object)

In [54]:
## ruta de base de datos
path_db = Path("C:/Users/PC/Desktop/Proyectos/Proyectos_Py/9.Tasas_Interes/data/base/tasas.db")

## abrir conexión a la base de datos
conn = sqlite3.connect(path_db)

In [48]:
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS tasa_dia (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    Periodo INTEGER,
    Fecha_Norm DATE,
    Bancos TEXT,
    Tipo_Tasa TEXT,
    tasa_interes FLOAT
)
""")

conn.commit()
# conn.close()

In [75]:
cursor.execute("DROP TABLE IF EXISTS prueba_tasas")
conn.commit()

In [ ]:
data_tasas.to_sql(
        "tasa_dia",
        conn,
        if_exists="replace", ## append, replace, fail
        index=False
    )

19080

In [58]:
query = """
SELECT DISTINCT Tipo_Tasa
FROM tasa_dia
"""

pd.read_sql_query(query, conn)

,Tipo_Tasa
0,Tarjetas de Crédito
1,Préstamos Revolventes
2,Consumo
3,Préstamos no Revolventes para automóviles
4,Préstamos no Revolventes para libre disponibi...
5,Préstamos no Revolventes para libre disponibi...
6,Préstamos hipotecarios para vivienda


In [65]:
query = """
SELECT *
FROM tasa_dia
WHERE Tipo_Tasa = 'Préstamos hipotecarios para vivienda'
AND Tasa_Interes IS NOT NULL
AND Fecha_Norm = (SELECT MAX(Fecha_Norm) FROM tasa_dia)
ORDER BY tasa_interes ASC
"""

tasa_hoy = pd.read_sql_query(query, conn)

tasa_hoy

,Periodo,Fecha_Norm,Bancos,Tipo_Tasa,tasa_interes
0,202606,2026-06-01 00:00:00,Scotiabank,Préstamos hipotecarios para vivienda,7.41
1,202606,2026-06-01 00:00:00,GNB,Préstamos hipotecarios para vivienda,7.44
2,202606,2026-06-01 00:00:00,BBVA,Préstamos hipotecarios para vivienda,7.74
3,202606,2026-06-01 00:00:00,Interbank,Préstamos hipotecarios para vivienda,7.76
4,202606,2026-06-01 00:00:00,Promedio,Préstamos hipotecarios para vivienda,7.86
5,202606,2026-06-01 00:00:00,Pichincha,Préstamos hipotecarios para vivienda,7.96
6,202606,2026-06-01 00:00:00,BIF,Préstamos hipotecarios para vivienda,8.00
7,202606,2026-06-01 00:00:00,Crédito,Préstamos hipotecarios para vivienda,8.07
8,202606,2026-06-01 00:00:00,Bancom,Préstamos hipotecarios para vivienda,9.66
9,202606,2026-06-01 00:00:00,Santander Cons. Bank,Préstamos hipotecarios para vivienda,11.00


In [72]:
tasa_result = {}
for i, row in tasa_hoy.iterrows():
    tasa_result[row['Bancos']] = row['tasa_interes']
    
tasa_result

{'Scotiabank': 7.41,
 'GNB': 7.44,
 'BBVA': 7.74,
 'Interbank': 7.76,
 'Promedio': 7.86,
 'Pichincha': 7.96,
 'BIF': 8.0,
 'Crédito': 8.07,
 'Bancom': 9.66,
 'Santander Cons. Bank': 11.0,
 'Mibanco': 15.88}